In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import scanpy as sc
from src.scmrdr.module import Integration

print("正在加载数据...")
adata = sc.read_h5ad("./experiments/BMMC_codes/feature_aligned_sampled.h5ad")
print(f"原始数据形状: {adata.shape}")

# === 新增步骤：筛选高变特征 ===
# 建议将特征数控制在 2000 左右，否则显存会爆
target_features = 2000 

if adata.shape[1] > target_features:
    print(f"特征数量过多 ({adata.shape[1]})，正在筛选前 {target_features} 个高变特征...")
    
    # 注意：如果是多模态数据，最好分别对 RNA 和 ATAC 筛选，或者如果数据已经预处理过，
    # 直接随机采样或取方差最大的前 2000 个
    # 这里使用简单的方差筛选作为示例：
    sc.pp.highly_variable_genes(adata, n_top_genes=target_features, subset=True)
    
    print(f"筛选后数据形状: {adata.shape}")
# ============================

# 2. 初始化模型
# 注意：你的数据里 layer 名字叫 'counts' (多了个s)，原代码是 'count'，这里要改一下
model = Integration(
    data=adata, 
    modality_key="modality", 
    layer="counts",       # <--- 重点：改成 counts
    batch_key="batch",    # 你的数据里有 batch 列，直接用
    feature_list=None,    # 你已经对齐了特征，这里填 None 即可
    distribution="ZINB"
)

# 3. 设置参数
model.setup(hidden_layers=[512,512], latent_dim_shared=20, latent_dim_specific=20, gamma=500, lambda_adv=200, dropout_rate=0.2)

# 4. 开始训练
print("开始训练模型...")
model.train(epoch_num=300, batch_size=128, lr=1e-3, adaptlr=False, num_warmup=0,
            early_stopping=False, valid_prop=0.1, weighted=False, patience=10)

# 5. 推断与获取结果
print("正在推断潜在空间...")
model.inference(n_samples=1, update=True, returns=False)
adata = model.get_adata() # 结果存储在 adata.obsm["latent_shared"]

# 6. 可视化
print("正在绘图...")
sc.pp.neighbors(adata, use_rep="latent_shared")
sc.tl.umap(adata)
sc.pl.umap(
    adata,
    color=["modality", "celltype", "batch"], # 注意：你的数据里叫 celltype
    size=2, wspace=0.5
)

正在加载数据...


/root/miniconda3/envs/scMRDR/lib/python3.11/site-packages/anndata/_core/anndata.py:1796: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


原始数据形状: (58868, 10170)
特征数量过多 (10170)，正在筛选前 2000 个高变特征...


/root/miniconda3/envs/scMRDR/lib/python3.11/site-packages/numba/np/ufunc/parallel.py:373: NumbaWarning: The TBB threading layer requires TBB version 2021 update 6 or later i.e., TBB_INTERFACE_VERSION >= 12060. Found TBB_INTERFACE_VERSION = 12050. The TBB threading layer is disabled.
  warnings.warn(problem)
/root/miniconda3/envs/scMRDR/lib/python3.11/site-packages/anndata/_core/anndata.py:1796: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


筛选后数据形状: (58868, 2000)
using cuda
开始训练模型...
Training start!
epoch 1: loss = 1198.6602, Diff_loss = 1369.0148, preserve_loss = 0.0567, adv_loss = -0.9935, discri_loss = 0.7369
epoch 2: loss = 348.3282, Diff_loss = 550.4505, preserve_loss = 0.0018, adv_loss = -1.0150, discri_loss = 0.7807
epoch 3: loss = 278.3207, Diff_loss = 484.3471, preserve_loss = 0.0016, adv_loss = -1.0340, discri_loss = 0.7642
epoch 4: loss = 240.9570, Diff_loss = 444.0504, preserve_loss = 0.0019, adv_loss = -1.0203, discri_loss = 0.6928
epoch 5: loss = 227.1686, Diff_loss = 431.7546, preserve_loss = 0.0028, adv_loss = -1.0299, discri_loss = 0.7234
epoch 6: loss = 210.9077, Diff_loss = 417.1341, preserve_loss = 0.0029, adv_loss = -1.0384, discri_loss = 0.6850
epoch 7: loss = 202.5915, Diff_loss = 408.9724, preserve_loss = 0.0028, adv_loss = -1.0390, discri_loss = 0.6588
epoch 8: loss = 195.9724, Diff_loss = 402.4936, preserve_loss = 0.0025, adv_loss = -1.0388, discri_loss = 0.6406
epoch 9: loss = 188.0075, Diff_los

KeyboardInterrupt: 

In [3]:
import numpy as np
import pandas as pd
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from sklearn.cluster import KMeans

# 1. 提取模型生成的共享潜在空间特征
# 根据你的代码 model.get_adata() 之后，特征在 "latent_shared"
latent_features = adata.obsm["latent_shared"]

# 2. 获取真实的细胞类型标签（参考答案）
true_labels = adata.obs["celltype"]

# 3. 计算真实的细胞类型数量 (K值)
n_clusters = len(true_labels.unique())

# 4. 使用 KMeans 进行聚类（这是生成论文表格的标准做法）
print(f"正在进行 KMeans 聚类 (K={n_clusters})...")
kmeans = KMeans(n_clusters=n_clusters, n_init=20, random_state=42)
pred_labels = kmeans.fit_predict(latent_features)

# 5. 计算得分
nmi_score = normalized_mutual_info_score(true_labels, pred_labels)
ari_score = adjusted_rand_score(true_labels, pred_labels)

# 6. 打印结果（这就是你想要的那张表里的数字）
print("\n" + "="*40)
print(f"{'指标名称':<15} | {'得分':<10}")
print("-" * 40)
print(f"{'KMeans NMI':<15} | {nmi_score:.4f}")
print(f"{'KMeans ARI':<15} | {ari_score:.4f}")
print("="*40)

正在进行 KMeans 聚类 (K=10)...

指标名称            | 得分        
----------------------------------------
KMeans NMI      | 0.0587
KMeans ARI      | 0.0319


In [2]:
!find /root -name "module.py"

/root/autodl-tmp/scMRDR/src/scMRDR/module.py
/root/miniconda3/envs/multiome/lib/python3.10/site-packages/debugpy/_vendored/pydevd/pydevd_attach_to_process/winappdbg/module.py
/root/miniconda3/envs/multiome/lib/python3.10/site-packages/jedi/inference/value/module.py
/root/miniconda3/lib/python3.12/site-packages/debugpy/_vendored/pydevd/pydevd_attach_to_process/winappdbg/module.py
/root/miniconda3/lib/python3.12/site-packages/jedi/inference/value/module.py
/root/miniconda3/lib/python3.12/site-packages/torch/nn/modules/module.py
/root/micromamba/envs/multiome310/lib/python3.10/site-packages/llvmlite/binding/module.py
/root/micromamba/envs/multiome310/lib/python3.10/site-packages/llvmlite/ir/module.py
/root/micromamba/envs/multiome310/lib/python3.10/site-packages/debugpy/_vendored/pydevd/pydevd_attach_to_process/winappdbg/module.py
/root/micromamba/envs/multiome310/lib/python3.10/site-packages/jedi/inference/value/module.py
/root/micromamba/pkgs/llvmlite-0.40.1-py310h1b8f574_0/lib/python3.